# VisionTracker Remote Server — Kaggle Setup

This notebook sets up a self-hosted vision-language model as a FastAPI server on Kaggle's free GPU.

## Features
- More GPU RAM (30GB vs Colab's 16GB)
- Supports GGUF and safetensors formats (<10GB)

## Prerequisites
1. Create a free Kaggle account at https://www.kaggle.com
2. Create a free ngrok account at https://dashboard.ngrok.com/signup
3. Add ngrok authtoken as Kaggle secret: Right sidebar → Add-ons → Secrets → NGROK_AUTHTOKEN

## Cell 1: Configure Model

In [ ]:
# CONFIGURATION
MODEL_FORMAT = "gguf"  # "gguf" or "safetensors"
MODEL_PATH = "https://huggingface.co/lmstudio-community/gemma-3-4b-it-GGUF/resolve/main/gemma-3-4b-it-Q4_K_M.gguf"
MODEL_FILENAME = "gemma-3-4b-it-Q4_K_M.gguf"
N_CTX = 4096
N_GPU_LAYERS = -1
print(f"Config: format={MODEL_FORMAT}, model={MODEL_PATH}")

## Cell 2: Install Dependencies

In [ ]:
!nvidia-smi
!pip install -q fastapi uvicorn python-multipart pyngrok pillow pydantic
if MODEL_FORMAT == "gguf":
    !CMAKE_ARGS="-DLLAMA_CUDA=on" pip install -q llama-cpp-python
else:
    !pip install -q transformers accelerate torch
print("Dependencies installed!")

## Cell 3: Configure ngrok

In [ ]:
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
try:
    NGROK_TOKEN = UserSecretsClient().get_secret("NGROK_AUTHTOKEN")
    print("Loaded NGROK_AUTHTOKEN from secrets")
except:
    NGROK_TOKEN = input("Enter ngrok authtoken: ").strip()
if not NGROK_TOKEN:
    raise ValueError("ngrok authtoken required!")
ngrok.set_auth_token(NGROK_TOKEN)
print("ngrok configured!")

## Cell 4: Download/Load Model

In [ ]:
import os
if MODEL_FORMAT == "gguf" and MODEL_PATH.startswith("http"):
    LOCAL_MODEL_PATH = f"/kaggle/working/{MODEL_FILENAME}"
    if not os.path.exists(LOCAL_MODEL_PATH):
        print(f"Downloading {MODEL_FILENAME}...")
        !wget -q --show-progress {MODEL_PATH} -O {LOCAL_MODEL_PATH}
    else:
        print("Model exists")
    print(f"Size: {os.path.getsize(LOCAL_MODEL_PATH)/1e9:.2f} GB")
else:
    LOCAL_MODEL_PATH = MODEL_PATH
    print(f"Using: {LOCAL_MODEL_PATH}")

## Cell 5: Load Model

In [ ]:
if MODEL_FORMAT == "gguf":
    from llama_cpp import Llama
    print("Loading GGUF model...")
    llm = Llama(model_path=LOCAL_MODEL_PATH, n_ctx=N_CTX, n_gpu_layers=N_GPU_LAYERS, verbose=False)
    _ = llm("Say 'ready'", max_tokens=10)
    print("Model loaded!")
else:
    from transformers import AutoProcessor, AutoModelForVision2Seq
    import torch
    device = "cuda" if torch.cuda.is_available() else "cpu"
    processor = AutoProcessor.from_pretrained(MODEL_PATH)
    model = AutoModelForVision2Seq.from_pretrained(
        MODEL_PATH, torch_dtype=torch.float16 if device=="cuda" else torch.float32,
        device_map="auto" if device=="cuda" else None)
    if device == "cpu":
        model = model.to("cpu")
    print(f"Model loaded on {device}!")

## Cell 6: Create FastAPI Server

In [ ]:
from fastapi import FastAPI, HTTPException, Header
from pydantic import BaseModel
from typing import List, Optional
from PIL import Image
import io, base64, re

app = FastAPI(title="VisionTracker Remote Server", version="2.0.0")

try:
    from kaggle_secrets import UserSecretsClient
    SERVER_API_KEY = UserSecretsClient().get_secret("SERVER_API_KEY")
except:
    SERVER_API_KEY = None

class IdentifyRequest(BaseModel):
    annotated_image: str
    track_ids: List[int]

def create_prompt(n):
    if n == 1:
        return "Describe the object inside the colored box in ONE sentence (<= 12 words)."
    numbered = chr(10).join([f"{i+1}. <desc>" for i in range(n)])
    return f"Describe each object in colored boxes. One sentence each. Format:\n{numbered}"

def parse_response(text, n, track_ids):
    pattern = re.compile(r"^\s*(\d+)[.)]\s*(.+)$")
    numbered = {}
    for line in text.splitlines():
        m = pattern.match(line)
        if m:
            numbered[int(m.group(1))-1] = m.group(2).strip()
    return [numbered.get(i, f"object #{track_ids[i]}") for i in range(n)]

def identify_gguf(image_b64, track_ids):
    n = len(track_ids)
    content = [{"type": "text", "text": create_prompt(n)}]
    content.append({"type": "image_url", "image_url": {"url": f"data:image/jpeg;base64,{image_b64}"}})
    resp = llm.create_chat_completion(
        messages=[{"role": "user", "content": content}],
        max_tokens=100*n, temperature=0.2, stop=["</s>"])
    return parse_response(resp["choices"][0]["message"]["content"].strip(), n, track_ids)

def identify_safetensors(image_b64, track_ids):
    import torch
    n = len(track_ids)
    img = Image.open(io.BytesIO(base64.b64decode(image_b64))).convert("RGB")
    device = "cuda" if torch.cuda.is_available() else "cpu"
    msgs = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": create_prompt(n)}]}]
    txt = processor.apply_chat_template(msgs, add_generation_prompt=True)
    inputs = processor(images=img, text=txt, return_tensors="pt").to(device)
    with torch.no_grad():
        outs = model.generate(**inputs, max_new_tokens=100*n, temperature=0.2, do_sample=True)
    return parse_response(processor.batch_decode(outs, skip_special_tokens=True)[0], n, track_ids)

@app.get("/health")
async def health():
    return {"status": "healthy", "format": MODEL_FORMAT, "model": str(MODEL_PATH)}

@app.post("/identify")
async def identify(req: IdentifyRequest, auth: Optional[str] = Header(None)):
    if SERVER_API_KEY and auth != f"Bearer {SERVER_API_KEY}":
        raise HTTPException(401, "Invalid API key")
    if not req.track_ids:
        return {"results": []}
    descs = identify_gguf(req.annotated_image, req.track_ids) if MODEL_FORMAT == "gguf" else identify_safetensors(req.annotated_image, req.track_ids)
    return {"results": [{"track_id": tid, "description": d} for tid, d in zip(req.track_ids, descs)]}

print("App created!")

## Cell 7: Start Server

In [ ]:
import nest_asyncio, uvicorn
nest_asyncio.apply()
url = ngrok.connect(8000, "http")
print("="*60)
print("KAGGLE SERVER RUNNING!")
print("="*60)
print(f"URL: {url}")
print(f"Health: {url}/health")
print(f"Identify: {url}/identify")
print("="*60)
print(f"\nUsage: python main.py --remote-url {url}")
uvicorn.run(app, host="0.0.0.0", port=8000, log_level="info")